# 1. Data Collection

Before any investment decision can be made, we need a complete picture of
the ETF universe available to a UK ISA investor. This chapter casts the net
as wide as possible — scraping every eligible UCITS ETF across four asset
classes — so the screening in the next chapter is working from the full
population, not an arbitrary shortlist.

Four asset classes are covered:

| Asset class | Scope | Output |
|---|---|---|
| **Equities** | Distributing UCITS ETFs, per country/region | One CSV per region |
| **Bonds** | Distributing UCITS bond ETFs, per currency/region | One CSV per region |
| **Precious metals** | Physical gold, silver, platinum ETCs | Single global CSV |
| **Commodities** | Broad commodity index ETCs | Single global CSV |

Equities and bonds are broken down by geography because the portfolio holds
regional sleeves (UK, APAC, EMEA, Emerging Markets). Precious metals and
commodities have no meaningful regional split — one global screen captures
everything.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json

import pandas as pd
import numpy as np
import justetf_scraping

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.database import save_raw_etf_data

## Defining the investment universe

The regional split for equities and bonds is anchored to **economic weight**,
not political convenience. We pull each country's latest GDP from the World
Bank so the regions we scrape reflect where actual economic activity sits.
Countries are also classified by their [MSCI](../content/99_glossary.md#pipeline-terms)
market status (Developed / Emerging), which determines the benchmark and
screening thresholds applied later.

Precious metals and commodities skip this step — they are sourced from a
single global screen.

In [2]:
import requests
import pandas as pd
from datetime import datetime

# Fetch latest GDP data dynamically from the World Bank API
def fetch_gdp_data():
    current_year = datetime.now().year
    
    # We query the last 2 years (e.g., 2022:2024) to ensure we get the most recently available data
    # as some countries might not have reported the current year's GDP yet.
    # The World Bank API naturally returns the most recent non-null records first when queried by a date range.
    url = f"https://api.worldbank.org/v2/country/all/indicator/NY.GDP.MKTP.CD?format=json&date={current_year-2}:{current_year}&per_page=1000"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        gdp_mapping = {}
        if len(data) > 1:
            for record in data[1]:
                country_id = record['country']['id']
                # If we haven't seen an entry for this country yet, and it's not None, add it.
                # Since the API returns data sorted descending by year, the first non-null is the most recent.
                if record.get('value') is not None and country_id not in gdp_mapping:
                    gdp_mapping[country_id] = {
                        'gdp': record['value'],
                        'year': record['date']
                    }
        return gdp_mapping
    except Exception as e:
        print(f"Error fetching GDP data: {e}")
        return {}

gdp_dict = fetch_gdp_data()

def format_gdp(val):
    if pd.isna(val) or val is None:
        return "N/A"
    if val >= 1e12:
        return f"${val/1e12:.3f} trillion"
    elif val >= 1e9:
        return f"${val/1e9:.3f} billion"
    else:
        return f"${val:,.0f}"

# Base country definitions without GDP
base_country_data = [
    ["United States", "Developed", "AmericasandUK", "US", "USD"],
    ["Canada", "Developed", "AmericasandUK", "CA", "CAD"],
    ["United Kingdom", "Developed", "AmericasandUK", "GB", "GBP"],
    ["Japan", "Developed", "APAC", "JP", "JPY"],
    ["Australia", "Developed", "APAC", "AU", "AUD"],
    ["Germany", "Developed", "EMEA", "DE", "EUR"],
    ["France", "Developed", "EMEA", "FR", "EUR"],
    ["Italy", "Developed", "EMEA", "IT", "EUR"],
    ["Spain", "Developed", "EMEA", "ES", "EUR"],
    ["Netherlands", "Developed", "EMEA", "NL", "EUR"],
    ["Switzerland", "Developed", "EMEA", "CH", "CHF"],
    ["Brazil", "Emerging", "Americas", "BR", "BRL"],
    ["Mexico", "Emerging", "Americas", "MX", "MXN"],
    ["China", "Emerging", "APACandEMEA", "CN", "CNY"],
    ["India", "Emerging", "APACandEMEA", "IN", "INR"],
    ["South Korea", "Emerging", "APACandEMEA", "KR", "KRW"],
    ["Indonesia", "Emerging", "APACandEMEA", "ID", "IDR"]
]

columns = ["Country", "MSCI", "Region", "Short_name", "Currency"]
country_df = pd.DataFrame(base_country_data, columns=columns)

# Map GDP dynamically
country_df["Latest GDP"] = country_df["Short_name"].apply(lambda x: gdp_dict.get(x, {}).get('gdp'))
country_df["GDP Year"] = country_df["Short_name"].apply(lambda x: gdp_dict.get(x, {}).get('year'))
country_df["Latest GDP"] = country_df["Latest GDP"].apply(format_gdp)

# Rename columns to reflect that it's the latest data
columns_updated = ["Country", "Latest GDP", "GDP Year", "MSCI", "Region", "Short_name", "Currency"]

# Reorder columns
country_df = country_df[columns_updated]

# Sort by MSCI then region
country_df.sort_values(by=["MSCI", "Region", "Latest GDP",], inplace=True)
country_df.reset_index(drop=True, inplace=True)

country_df



Error fetching GDP data: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=10)


,Country,Latest GDP,GDP Year,MSCI,Region,Short_name,Currency
0,Japan,N/A,None,Developed,APAC,JP,JPY
1,Australia,N/A,None,Developed,APAC,AU,AUD
2,United States,N/A,None,Developed,AmericasandUK,US,USD
3,Canada,N/A,None,Developed,AmericasandUK,CA,CAD
4,United Kingdom,N/A,None,Developed,AmericasandUK,GB,GBP
5,Germany,N/A,None,Developed,EMEA,DE,EUR
6,France,N/A,None,Developed,EMEA,FR,EUR
7,Italy,N/A,None,Developed,EMEA,IT,EUR
8,Spain,N/A,None,Developed,EMEA,ES,EUR
9,Netherlands,N/A,None,Developed,EMEA,NL,EUR


## Collecting ETF data

With the regional universe defined, we scrape [JustETF](https://www.justetf.com)
for each combination of asset class and region. JustETF is used purely as an
**ETF attribute database** — ticker, fund size, [TER](../content/99_glossary.md#costs),
distribution policy, benchmark index, domicile. Prices come from a separate
provider in later notebooks.

Each scrape result is stored locally so the rest of the pipeline can run
without repeating the network requests.

In [3]:
import traceback
import warnings

# Define asset classes to collect
# class-preciousMetals and class-commodities are scraped globally (no country loop)
asset_classes = [
    "class-equity",
    "class-bonds",
    "class-preciousMetals",
    "class-commodities",
]

# --- Global-only asset classes (no country/currency iteration) ---
GLOBAL_ASSET_CLASSES = {"class-preciousMetals", "class-commodities"}

# Group countries by region for data organization
region_countries = country_df.groupby('Region')['Short_name'].apply(list).to_dict()

# Create a reverse mapping of country to region
country_to_region = dict(zip(country_df['Short_name'], country_df['Region']))
country_to_country_name = dict(zip(country_df['Short_name'], country_df['Country']))
country_to_msci = dict(zip(country_df['Short_name'], country_df['MSCI']))

# Create a mapping of MSCI and Region to Category
msci_region_to_category = {
    ('Developed', 'AmericasandUK'): 'Developed_AmericasandUK',
    ('Developed', 'EMEA'): 'Developed_EMEA',
    ('Developed', 'APAC'): 'Developed_APAC',
    ('Emerging', 'Americas'): 'Emerging_Americas',
    ('Emerging', 'APACandEMEA'): 'Emerging_APACandEMEA'
}

for asset_class in asset_classes:
    asset_class_name = asset_class.split("-", 1)[1]

    # ------------------------------------------------------------------
    # Global-only asset classes: single scrape, no country iteration
    # ------------------------------------------------------------------
    if asset_class in GLOBAL_ASSET_CLASSES:
        try:
            df = justetf_scraping.load_overview(asset_class=asset_class, local_country="GB")
            df['region'] = 'Global'
            df['country'] = 'Global'
            df['region_category'] = 'Global'

            filename = f'justetf_class-{asset_class_name}_global.csv'
            filepath = DATA_RAW / filename
            df.to_csv(filepath, index=False)
            print(f"Scraped {len(df)} ETFs for {asset_class_name} (Global) → {filename}")
        except Exception as e:
            print(f"Error scraping {asset_class} (Global): {e}")
            continue

        continue  # skip the country loop below

    # ------------------------------------------------------------------
    # Country/currency iteration for equity and bonds
    # ------------------------------------------------------------------
    for country in country_df['Short_name']:
        # --- Scrape + CSV save (fatal on failure) ---
        try:
            if asset_class == "class-equity":
                df = justetf_scraping.load_overview(asset_class=asset_class, country=country, local_country="GB")
            else:
                currency = country_df[country_df['Short_name'] == country]['Currency'].values[0]
                df = justetf_scraping.load_overview(asset_class=asset_class, instrument_currency=currency, local_country="GB")

            df['region'] = country_to_region[country]
            df['country'] = country_to_country_name[country]

            msci = country_to_msci[country]
            region = country_to_region[country]
            region_category = msci_region_to_category.get((msci, region), "Unknown")
            df['region_category'] = region_category

            # Save CSV (append + dedup by ticker)
            filename = f'justetf_{asset_class}_{region_category.lower()}.csv'.lower()
            filepath = DATA_RAW / filename

            if filepath.exists():
                existing_df = pd.read_csv(filepath)
                combined_df = pd.concat([existing_df, df], ignore_index=True).drop_duplicates(subset=['ticker'])
                combined_df.to_csv(filepath, index=False)
            else:
                df.to_csv(filepath, index=False)

            print(f"Processed {country} data for {asset_class} -> {region_category}")
        except Exception as e:
            print(f"Error scraping {asset_class} for {country}: {e}")
            continue

# --- End of scraping ---
# Now, load all the accumulated CSVs and reliably write them to the DB once per category
print("\n--- Syncing collected CSVs to SQLite Database ---")
for filepath in sorted(DATA_RAW.glob("justetf_class-*.csv")):
    csvl = filepath.name
    ac = get_asset_class_from_filename(csvl)
    rc = get_region_category_from_filename(csvl)
    df_merged = pd.read_csv(filepath)
    try:
        save_raw_etf_data(df_merged, ac, rc)
        print(f"[DB] Synced {len(df_merged)} ETFs for {ac}/{rc}")
    except Exception as e:
        warnings.warn(f"[DB] Could not save {ac}/{rc} to database: {e}")

Processed JP data for class-equity -> Developed_APAC
Processed AU data for class-equity -> Developed_APAC
Processed US data for class-equity -> Developed_AmericasandUK
Processed CA data for class-equity -> Developed_AmericasandUK
Processed GB data for class-equity -> Developed_AmericasandUK
Processed DE data for class-equity -> Developed_EMEA
Processed FR data for class-equity -> Developed_EMEA
Processed IT data for class-equity -> Developed_EMEA
Processed ES data for class-equity -> Developed_EMEA
Processed NL data for class-equity -> Developed_EMEA
Processed CH data for class-equity -> Developed_EMEA
Processed CN data for class-equity -> Emerging_APACandEMEA
Processed IN data for class-equity -> Emerging_APACandEMEA
Processed KR data for class-equity -> Emerging_APACandEMEA
Processed ID data for class-equity -> Emerging_APACandEMEA
Processed BR data for class-equity -> Emerging_Americas
Processed MX data for class-equity -> Emerging_Americas
Processed JP data for class-bonds -> Devel

## Summary

The table below shows how many ETFs were collected per asset class and
region. These raw counts are the starting population for the screening step —
most will be filtered out, but it is important to begin from the full universe
to avoid survivorship or cherry-picking bias.

In [5]:
summary_data = []
for filepath in sorted(DATA_RAW.glob("justetf_class-*.csv")):
    csvl = filepath.name
    asset_class = get_asset_class_from_filename(csvl)
    region_category = get_region_category_from_filename(csvl)
    df = pd.read_csv(filepath)
    summary_data.append({
        'Asset Class': asset_class.title(),
        'Category': region_category,
        'NumberofETFs': len(df)
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df.pivot_table(
    index='Category',
    columns='Asset Class',
    values=['NumberofETFs'],
    aggfunc='first'
))

NumberofETFs                                  
Asset Class                    Bonds Commodities Equity Preciousmetals
Category                                                              
developed_americasanduk        529.0         NaN  675.0            NaN
developed_apac                  13.0         NaN  135.0            NaN
developed_emea                 427.0         NaN   86.0            NaN
emerging_americas                1.0         NaN    7.0            NaN
emerging_apacandemea            25.0         NaN   82.0            NaN
global                           NaN       121.0    NaN           52.0